# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laspric/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research Question

Which content pages should be prioritized for review or refresh based on their search performance signals?

### Decision Supported

This analysis will help content teams identify and prioritize pages that may deserve review, refresh, improvement, or monitoring.

## 2. Data

### Data Source

This project will use the FlyRank ML Internship full warehouse release.

### Data Selection

I will select the tables and fields needed to study content refresh opportunities and search performance.

### Exclusions

Client-identifying information, private queries, credentials, raw exports, and label-derived or future-looking fields that could cause data leakage will be excluded from the modeling features.

### Public Safety

The analysis will use only safe, aggregated or pseudonymous data and will not disclose client names, domains, private queries, or confidential information.

In [2]:
!git clone https://github.com/Laspric/flyrank-internship-ml.git
%cd /content/flyrank-internship-ml

fatal: destination path 'flyrank-internship-ml' already exists and is not an empty directory.
/content/flyrank-internship-ml


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
# Columns that may contain the outcome/label
print("Potential label/leakage columns:")
print([
    "trend_direction",
    "trend_pct"
])

print("\nTheir values:")
print(df[["trend_direction", "trend_pct"]].head())

Potential label/leakage columns:
['trend_direction', 'trend_pct']

Their values:
  trend_direction  trend_pct
0            down      -41.4
1            down      -57.7
2            down      -60.9
3          stable      -13.8
4            down      -34.7


In [5]:
print("Potential label/leakage columns:")
print([
    "trend_direction",
    "trend_pct"
])

print("\nTheir values:")
print(df[["trend_direction", "trend_pct"]].head())

Potential label/leakage columns:
['trend_direction', 'trend_pct']

Their values:
  trend_direction  trend_pct
0            down      -41.4
1            down      -57.7
2            down      -60.9
3          stable      -13.8
4            down      -34.7


In [6]:
# Step 10 — Group the dataset columns

id_columns = [
    "content_id",
    "client_id"
]

possible_leakage = [
    "trend_direction",
    "trend_pct"
]

print("ID / grouping columns:")
print(id_columns)

print("\nPossible target / leakage columns:")
print(possible_leakage)

print("\nAll remaining columns:")
remaining_columns = [
    col for col in df.columns
    if col not in id_columns + possible_leakage
]

print(remaining_columns)
print("\nNumber of remaining columns:", len(remaining_columns))

ID / grouping columns:
['content_id', 'client_id']

Possible target / leakage columns:
['trend_direction', 'trend_pct']

All remaining columns:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Number of remaining columns: 40


In [7]:
# Step 11 — Inspect candidate features

candidate_columns = [
    col for col in df.columns
    if col not in ["content_id", "client_id", "trend_direction", "trend_pct"]
]

feature_info = pd.DataFrame({
    "column": candidate_columns,
    "dtype": [df[col].dtype for col in candidate_columns],
    "missing": [df[col].isna().sum() for col in candidate_columns],
    "unique_values": [df[col].nunique() for col in candidate_columns]
})

print(feature_info.to_string(index=False))

                column   dtype  missing  unique_values
         search_volume float64     2468             41
           competition float64     2468            101
     competition_level  object     2610              3
                   cpc float64     2468            915
          content_type  object        0              3
           main_intent  object     2374              4
            word_count float64     7699           5476
            char_count float64     7699          14839
         provider_used  object    21438              2
            model_used  object     5733              5
       impressions_90d   int64        0           9438
            clicks_90d   int64        0            477
         pageviews_90d   int64        0            856
          sessions_90d   int64        0            666
             users_90d   int64        0            644
  engaged_sessions_90d   int64        0             68
       ai_sessions_90d   int64        0             35
     scrol

In [8]:
# Step 12 — Create the target

df["declining_target"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(df["declining_target"].value_counts())

print("\nTarget proportions:")
print(df["declining_target"].value_counts(normalize=True))

Target distribution:
declining_target
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
declining_target
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [9]:
# Step 13 — Define the model features

features = [
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

print("Number of features:", len(features))
print("\nFeatures:")
print(features)

Number of features: 30

Features:
['search_volume', 'competition', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [10]:
# Step 14 — Check missing values in selected features

missing = df[features].isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Features with missing values:")
print(missing)

print("\nTotal missing values:", df[features].isna().sum().sum())

Features with missing values:
word_count       7699
char_count       7699
search_volume    2468
cpc              2468
competition      2468
main_intent      2374
scroll_rate       125
dtype: int64

Total missing values: 25301


In [11]:
# Step 15 — Identify numeric and categorical features

numeric_features = df[features].select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = df[features].select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 28
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features: 2
['content_type', 'main_intent']


In [12]:
# Step 16 — Create training and test sets

from sklearn.model_selection import train_test_split

X = df[features]
y = df["declining_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())

Training rows: 24000
Test rows: 6000

Training decline rate: 0.5420833333333334
Test decline rate: 0.542


In [13]:
# Step 17 — Build preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created.
Numeric features: 28
Categorical features: 2


In [15]:
# Step 18 — Train Logistic Regression model

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [16]:
# Cell 1 — Load data and create target

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the target:
# 1 = declining, 0 = not declining
df["declining_target"] = (df["trend_direction"] == "down").astype(int)

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["declining_target"].value_counts())

print("\nTarget proportions:")
print(df["declining_target"].value_counts(normalize=True))

Dataset shape: (30000, 45)

Target distribution:
declining_target
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
declining_target
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [17]:
# Cell 2 — Define features and train/test split

from sklearn.model_selection import train_test_split

features = [
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features]
y = df["declining_target"]

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Features:", len(features))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())

Features: 30
Numeric features: 28
Categorical features: 2
Training rows: 24000
Test rows: 6000
Training decline rate: 0.5420833333333334
Test decline rate: 0.542


In [18]:
# Cell 3 — Preprocessing and Logistic Regression

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Full model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Train
model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [19]:
# Step 19 — Predictions and evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Class predictions
y_pred = model.predict(X_test)

# Probability of being declining
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Model Results")
print("----------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nBase rate:")
print(f"{y_test.mean():.4f}")

Model Results
----------------
Accuracy : 0.8212
Precision: 0.8339
Recall   : 0.8367
F1 Score : 0.8353
ROC-AUC  : 0.9146

Base rate:
0.5420


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
